In [29]:
from settings import get_config
config = get_config()

from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

In [30]:
model = ChatOpenAI(model="gpt-5.4-mini", api_key=config.OPENAI_API_KEY)

In [31]:
def _web_search(query: str, gl: str, hl: str) -> str:
    wrapper = GoogleSerperAPIWrapper(
        serper_api_key=config.SERPER_API_KEY,
        gl=gl,
        hl=hl,
        type="search",
    )
    return wrapper.run(query)


@tool
def search_brazil(query: str) -> str:
    """Run an organic Google web search localized to Brazil (gl=br, hl=pt-br).
    Returns the top organic results — useful to find which products, brands or
    services are most cited and ranked in Brazil."""
    return _web_search(query, gl="br", hl="pt-br")


@tool
def search_usa(query: str) -> str:
    """Run an organic Google web search localized to the USA (gl=us, hl=en).
    Returns the top organic results — useful to find which products, brands or
    services are most cited and ranked in the USA."""
    return _web_search(query, gl="us", hl="en")


@tool
def search_spain(query: str) -> str:
    """Run an organic Google web search localized to Spain (gl=es, hl=es).
    Returns the top organic results — useful to find which products, brands or
    services are most cited and ranked in Spain."""
    return _web_search(query, gl="es", hl="es")


tools = [search_brazil, search_usa, search_spain]

In [32]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a market research agent. Your job is to find which products, "
        "brands or services are most cited and ranked organically on Google in "
        "three regions: Brazil, USA, and Spain.\n\n"
        "For every user question:\n"
        "1. Run the search in ALL THREE regions (search_brazil, search_usa, search_spain).\n"
        "2. From each region, extract the products/brands that appear most often "
        "   in the top organic results, listicles ('top 10', 'best of'), reviews "
        "   and comparison articles.\n"
        "3. Build a ranked list per region based on how frequently each option "
        "   shows up across the results.\n"
        "4. Then produce a consolidated cross-region ranking and highlight which "
        "   options dominate globally vs. only in a specific region.\n\n"
        "Always reason from the actual search results. Do not refuse just because "
        "a single query returns weak data — try varied query phrasings (e.g. "
        "'best X', 'top X 2025', 'X comparison', in the local language) before "
        "concluding the data is insufficient."
    ),
    #checkpointer=MemorySaver()
)

In [35]:
response = agent.invoke({
      "messages": [{
          "role": "user",
          "content": "Quais são os chatbots para WhatsApp mais bem ranqueados organicamente no Google? Pesquise em BR, EUA e Espanha e me dê um ranking consolidado."
      }]
  })
print(response["messages"][-1].content)


Pesquisei em **Brasil, EUA e Espanha** com variações como “best/ top WhatsApp chatbot 2025”, “comparison” e “business platforms”. Com base nos **resultados orgânicos que mais se repetiram** em listas, comparativos e reviews, aqui vai o ranking.

## Ranking por região

### 🇧🇷 Brasil
Os nomes que mais aparecem nos resultados:
1. **ManyChat**
2. **WATI**
3. **Kommo**
4. **Zenvia**
5. **SendPulse**
6. **Landbot**
7. **Tiledesk**
8. **JivoChat**
9. **HubSpot**
10. **Zendesk**

**Leitura do BR:**  
- **ManyChat** e **WATI** aparecem em várias listas de “melhores chatbots para WhatsApp”.
- **Kommo**, **Zenvia** e **SendPulse** também são recorrentes em listas brasileiras.
- Há forte presença de ferramentas com foco em vendas/CRM e atendimento omnichannel.

---

### 🇺🇸 USA
Os nomes mais citados:
1. **WATI**
2. **ManyChat**
3. **Botpress**
4. **Chatbase**
5. **Tidio**
6. **UChat**
7. **respond.io**
8. **Chatfuel**
9. **Gupshup**
10. **Trengo**

**Leitura do EUA:**  
- **WATI** lidera com muita 